In [ ]:
from matplotlib import pyplot as plt
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import seaborn as sns
import pandas as pd

sns.set_theme(style="darkgrid")
 
magdata = []
kronDiffData = []

In [ ]:
with open('/users/cdcook/VSP/datafiles/PSdata/xtetrans_1b_Exp4Final.csv', newline='') as w:
    data = list(csv.reader(w))
    data.pop(0)

In [ ]:
def colorSub(data, max):
    counter = 0
    RotseMag = []
    gPSFmag = []
    rPSFmag = []
    iPSFmag = []
    zPSFmag = []
    yPSFmag = []
    gKRONmag = []
    rKRONmag = []
    iKRONmag = []
    zKRONmag = []
    yKRONmag = []
    bitFlags = []
    pseudoBoloMag = []
    PanRotDiff = []
    kronDist = []
    counter = 0
    for n in range(max):
        if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0:
            counter = counter + 1
            RotseMag.append(float(data[n][4]))
            bitFlags.append(float(data[n][5]))
            gPSFmag.append(float(data[n][6]))
            rPSFmag.append(float(data[n][8]))
            iPSFmag.append(float(data[n][10]))
            zPSFmag.append(float(data[n][12]))
            yPSFmag.append(float(data[n][14]))
            gKRONmag.append(float(data[n][7]))
            rKRONmag.append(float(data[n][9]))
            iKRONmag.append(float(data[n][11]))
            zKRONmag.append(float(data[n][13]))
            yKRONmag.append(float(data[n][15]))
            gflux = pow(10, (float(data[n][6])+48.6)/-2.5)
            rflux = pow(10, (float(data[n][8])+48.6)/-2.5)
            iflux = pow(10, (float(data[n][10])+48.6)/-2.5)
            zflux = pow(10, (float(data[n][12])+48.6)/-2.5)
            yflux = pow(10, (float(data[n][14])+48.6)/-2.5)
            #totalFlux = (gflux*(7.058-5.454) + rflux*(5.454-4.347) + iflux*(4.347-3.68) + zflux*(3.68-3.26) +yflux*(3.26-3))/((7.058-3))
            #totalFlux = (gflux*(52.5) + rflux*(97.5) + iflux*(106.25) + zflux*(72.5) +yflux*(52.5))/((381.25))
            #totalFlux = (gflux*(551-414) + rflux*(689-550) + iflux*(819-690) + zflux*(922-818) +yflux*(1001-918))/((592))
            totalFlux = (gflux*(0.1212) + rflux*(0.1463) + iflux*(0.1435) + zflux*(0.098) +yflux*(.0393))/((0.5483))
            logpart = math.log(totalFlux/3631e-23, 10)
            pseudoBoloMag.append(-2.5*logpart)
            PanRotDiff.append(-2.5*logpart - float(data[n][4]))
            kronDist.append(float(data[n][4])-float(data[n][7]))
    data={'RotseMag':RotseMag, 'bitFlags':bitFlags, 'gPSFmag':gPSFmag, 'rPSFmag':rPSFmag, 'iPSFmag':iPSFmag, 'zPSFmag':zPSFmag, 'yPSFmag':yPSFmag, 'gKRONmag':gKRONmag, 'rKRONmag':rKRONmag, 'iKRONmag':iKRONmag, 'zKRONmag':zKRONmag, 'yKRONmag':yKRONmag, 'pseudoBoloMag':pseudoBoloMag, 'PanRotDiff':PanRotDiff, 'kronDist':kronDist}
    combinedDF = pd.DataFrame(data)
    return combinedDF    

In [ ]:
Title = 'Kron Difference - g'
kronBand = 6
bitFlagsTF = False
bitFlags = '00000000'
max = 8090

if (kronBand == 6):
    psfName = 'gPSFmag'
    kronName = 'gKRONmag'
if (kronBand == 8):
    psfName = 'rPSFmag'
    kronName = 'rKRONmag'
if (kronBand == 10):
    psfName = 'iPSFmag'
    kronName = 'iKRONmag'
if (kronBand == 12):
    psfName = 'zPSFmag'
    kronName = 'zKRONmag'
if (kronBand == 14):
    psfName = 'yPSFmag'
    kronName = 'yKRONmag'

In [ ]:
dataDF = colorSub(data, 8090)

In [ ]:
kronDist = 0.5 # Condition 1: psfName - kronName should be less than kronDist
condition1 = abs(dataDF[psfName] - dataDF[kronName]) < kronDist
# Apply both conditions to the DataFrame
dataDF = dataDF[condition1]

In [ ]:
plt.figure(figsize=(18,9))
sns.scatterplot(x = psfName, y = kronDist, data=dataDF)
print("slope: ", params[0][0])
print("AB offset: ", params[0][1])
print("Total Count: ", counter)
print("Above red line: ", gaxCount)   
plt.xlabel('PanSTARRS PSFMag', fontsize=12)
plt.ylabel('PanSTARRS PSFMag-kronMag', fontsize=12)
#plt.title(kronName + ' -- Kron Difference', fontsize = 20)
plt.title(kronName + ' -- Kron Difference w/ no rotse e-flags', fontsize = 20)
plt.show()